# Глубокий анализ признаков

## Цель ноутбука
- Проанализировать распределения числовых признаков
- Изучить выбросы с помощью boxplot
- Проанализировать взаимосвязь признаков с целевой переменной
- Построить корреляционную матрицу

## Используемые данные
- Очищенный датасет из ноутбука 01

## Основные выводы
- Некоторые признаки имеют выбросы (Age, Balance)
- Категориальные признаки показывают разную склонность к оттоку
- Корреляция между признаками слабая

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = BASE_DIR / 'data' / 'raw' / 'Churn_Modelling.csv'

drop_columns = ['RowNumber', 'CustomerId', 'Surname']
target = 'Exited'

numeric_features = [
    'CreditScore', 'Age', 'Tenure', 'Balance',
    'NumOfProducts', 'EstimatedSalary'
]

categorical_features = ['Geography', 'Gender']
binary_features = ['HasCrCard', 'IsActiveMember']

df = pd.read_csv(DATA_PATH)
df_clean = df.drop(columns=drop_columns).drop_duplicates().copy()

## 1. Анализ числовых признаков

In [ ]:
df_clean[numeric_features].hist(bins=20, figsize=(14, 10))
plt.suptitle('Распределения числовых признаков', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
for col in numeric_features:
    plt.figure()
    sns.boxplot(x=df_clean[col])
    plt.title(f'Boxplot: {col}')
    plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, col in enumerate(numeric_features):
    sns.boxplot(data=df_clean, x=target, y=col, ax=axes[idx])
    axes[idx].set_title(f'{col} по Exited')

plt.tight_layout()
plt.show()

## 2. Корреляционный анализ

In [ ]:
corr = df_clean[numeric_features + [target]].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Корреляционная матрица числовых признаков')
plt.show()

## 3. Анализ категориальных и бинарных признаков

In [ ]:
for col in categorical_features + binary_features:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.countplot(data=df_clean, x=col, ax=ax1)
    ax1.set_title(f'Распределение {col}')
    ax1.set_xlabel(col)
    ax1.set_ylabel('Количество')
    
    sns.countplot(data=df_clean, x=col, hue=target, ax=ax2)
    ax2.set_title(f'Распределение {col} относительно Exited')
    ax2.set_xlabel(col)
    ax2.set_ylabel('Количество')
    ax2.legend(title='Exited', labels=['Нет', 'Да'])
    
    plt.tight_layout()
    plt.show()

## 4. Выводы по анализу признаков

- `Age` имеет значительный разброс и выбросы в сторону старшего возраста
- `Balance` имеет бимодальное распределение
- Клиенты с большим балансом чаще уходят
- Активные клиенты (`IsActiveMember`) реже уходят
- Количество продуктов (`NumOfProducts`) влияет на отток
- Корреляция между признаками слабая, что хорошо для моделей